In [ ]:
# 1. Tải file ZIP từ link chia sẻ công khai (Dùng ID bạn đã cung cấp)
!pip install --upgrade gdown --quiet
import gdown

# ID file ZIP từ link của bạn
import os
from dotenv import load_dotenv
load_dotenv()

zip_id = os.getenv('ZIP_ID')
gdown.download(id=zip_id, output='/content/Ai_Deepfake.zip', quiet=False)

# 2. Giải nén folder dự án
# Lệnh này sẽ giải nén vào thư mục /content/Ai_Deepfake
!unzip -q /content/Ai_Deepfake.zip -d /content/

# 3. Di chuyển vào thư mục dự án và cài đặt thư viện
# Lưu ý: Kiểm tra xem folder sau khi giải nén tên là gì (thường là Ai_Deepfake hoặc SelfBlendedImages)
%cd /content/Ai_Deepfake
!pip install efficientnet_pytorch albumentations==1.3.0 opencv-python dlib tqdm scikit-learn python-dotenv retinaface-pytorch fastapi uvicorn python-multipart playwright nest_asyncio python-whois beautifulsoup4 imagehash --quiet

# 4. Cài đặt các thành phần hệ thống cần thiết
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb
!playwright install chromium
!playwright install-deps

# 5. Thiết lập đường dẫn hệ thống (Path)
import sys, os
sys.path.append(os.getcwd())
sys.path.append(os.path.join(os.getcwd(), 'src', 'inference'))

print("\n✅ Môi trường đã được thiết lập thành công từ file ZIP!")

In [ ]:
import torch
import cv2
import numpy as np
import subprocess
import time
import re
import requests
import signal
import threading
import os
import nest_asyncio
import urllib.parse
import asyncio
import whois
import imagehash
import uuid
from datetime import datetime
from bs4 import BeautifulSoup
from PIL import Image as PILImage
from fastapi import FastAPI, WebSocket
import uvicorn
from IPython.display import display, clear_output
from playwright.async_api import async_playwright

# 1. THIẾT LẬP ĐƯỜNG DẪN TUYỆT ĐỐI (QUAN TRỌNG)
# Việc này đảm bảo dù bạn đang đứng ở đâu, code vẫn tìm thấy Model và Module
BASE_DIR = "/content/Ai_Deepfake"
os.chdir(BASE_DIR) # Ép Python làm việc trong thư mục này

import sys
sys.path.append(BASE_DIR)
sys.path.append(os.path.join(BASE_DIR, 'src'))
sys.path.append(os.path.join(BASE_DIR, 'src', 'inference'))

# Bây giờ mới import các file cục bộ để tránh lỗi ModuleNotFoundError
from model import Detector
from preprocess import extract_face
from retinaface.pre_trained_models import get_model

# Áp dụng bản vá cho môi trường Jupyter/Colab
nest_asyncio.apply()

# ==========================================
# CẤU HÌNH & BIẾN TOÀN CỤC
# ==========================================
FIREBASE_DB_URL = os.getenv('FIREBASE_DB_URL')
RESTART_INTERVAL = 3600
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

app = FastAPI()
model = None
face_detector = None

# ==========================================
# 1. KHỞI TẠO MODEL AI (DEEPFAKE)
# ==========================================
def load_models():
    global model, face_detector
    print("⏳ Đang tải AI Models vào GPU...")
    # Sử dụng đường dẫn tuyệt đối để chắc chắn 100%
    weight_path = os.path.join(BASE_DIR, "weights/FFc23.tar")

    if not os.path.exists(weight_path):
        print(f"❌ LỖI: Không tìm thấy file weights tại {weight_path}")
        return

    model = Detector().to(device)
    cnn_sd = torch.load(weight_path, map_location=device)["model"]
    model.load_state_dict(cnn_sd)
    model.eval()

    face_detector = get_model("resnet50_2020-07-20", max_size=2048, device=device)
    face_detector.eval()
    print("✅ Models đã sẵn sàng trên GPU!")

# ... (Các hàm predict_deepfake, analyze_dom_and_phash, check_chongluadao GIỮ NGUYÊN) ...

def predict_deepfake(frame_bgr):
    try:
        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        face_list = extract_face(frame_rgb, face_detector)

        if face_list is None or len(face_list) == 0:
            return "NO_FACE", 0.0

        with torch.no_grad():
            img_tensor = torch.tensor(face_list).to(device).float() / 255.0
            pred = model(img_tensor).softmax(1)[:, 1].cpu().data.numpy()
            score = float(np.max(pred))

        label = "DEEPFAKE" if score > 0.8 else "REAL"
        return label, score * 100
    except Exception as e:
        return "ERROR", 0.0

def clean_domain(url):
    if not url.startswith('http'):
        url = 'http://' + url
    parsed_uri = urllib.parse.urlparse(url)
    return parsed_uri.netloc.replace('www.', '')

def format_url(domain):
    return domain if domain.startswith("http") else f"https://{domain}"

def get_domain_age_sync(input_url):
    domain = clean_domain(input_url)
    try:
        domain_info = whois.whois(domain)
        creation_date = domain_info.creation_date
        if isinstance(creation_date, list):
            creation_date = creation_date[0]

        if creation_date:
            creation_date = creation_date.replace(tzinfo=None)
            age_days = (datetime.now() - creation_date).days
            return age_days
    except Exception as e:
        print(f"[!] Lỗi WHOIS: {str(e)}")
    return -1

async def analyze_dom_and_phash(url):
    results = {"phash": None, "risk_flags": [], "suspicious_forms_count": 0}
    screenshot_path = f"temp_screenshot_{uuid.uuid4().hex}.png"

    async with async_playwright() as p:
        try:
            browser = await p.chromium.launch(headless=True, args=['--no-sandbox', '--disable-setuid-sandbox'])
            context = await browser.new_context(
                user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
            )
            page = await context.new_page()
            await page.goto(url, timeout=20000, wait_until="networkidle")

            await page.screenshot(path=screenshot_path)
            hash_val = imagehash.phash(PILImage.open(screenshot_path))
            results["phash"] = str(hash_val)

            html_content = await page.content()
            soup = BeautifulSoup(html_content, 'html.parser')
            current_domain = clean_domain(url)

            forms = soup.find_all('form')
            for form in forms:
                action = form.get('action')
                if action:
                    action_domain = clean_domain(action)
                    if action_domain and action_domain != current_domain:
                        results["risk_flags"].append("Form sends data to an unauthorized external domain")
                        results["suspicious_forms_count"] += 1

            pass_inputs = soup.find_all('input', type='password')
            if pass_inputs:
                results["risk_flags"].append("Page requests password input (High phishing risk if unexpected)")
                results["suspicious_forms_count"] += len(pass_inputs)

            iframes = soup.find_all('iframe')
            for iframe in iframes:
                style = iframe.get('style', '').lower()
                if 'display:none' in style or 'display: none' in style or 'opacity:0' in style or 'visibility:hidden' in style:
                    results["risk_flags"].append("Hidden Iframe detected (Sign of malware injection or user tracking)")

            body_text = soup.get_text().lower()
            sensitive_keywords = ["xác minh tài khoản", "cung cấp mã otp", "số cccd", "mã bảo mật cvv", "tài khoản bị khóa"]
            found_keys = [kw for kw in sensitive_keywords if kw in body_text]
            if found_keys:
                results["risk_flags"].append(f"Phishing/threatening text detected: {', '.join(found_keys)}")

            results["risk_flags"] = list(set(results["risk_flags"]))

        except Exception as e:
            results["risk_flags"].append("Cannot access page (Error or Timeout)")
        finally:
            if 'browser' in locals():
                await browser.close()
            if os.path.exists(screenshot_path):
                os.remove(screenshot_path)
    return results

async def check_chongluadao(domain_to_check):
    target_url = format_url(domain_to_check)
    encoded_target_url = urllib.parse.quote(target_url, safe='')
    analyze_url = f"https://chongluadao.vn/analyze/?q={encoded_target_url}&type=url"
    danger_keywords = ["nguy hiểm", "cảnh báo", "phishing", "malware", "scam", "lừa đảo"]
    safe_keywords = ["an toàn", "tin cậy", "chính đáng"]

    async with async_playwright() as p:
        try:
            browser = await p.chromium.launch(headless=True, args=['--no-sandbox', '--disable-setuid-sandbox'])
            context = await browser.new_context()
            page = await context.new_page()
            await page.goto(analyze_url, wait_until="networkidle", timeout=20000)
            await page.locator('xpath=//*[@id="app"]/div/section/form/button').click()
            target_xpath = '//*[@id="app"]/div/div[2]/section/div[1]/article[1]/div[1]'
            await page.wait_for_selector(f'xpath={target_xpath}', timeout=15000)
            main_result = ""
            for _ in range(8):
                main_result = await page.locator(f'xpath={target_xpath}').inner_text()
                if "Đang tải" not in main_result and len(main_result.strip()) > 0:
                    break
                await asyncio.sleep(1)
            main_result_lower = main_result.lower()
            if any(key in main_result_lower for key in danger_keywords):
                await browser.close()
                return "Dangerous"
            if any(key in main_result_lower for key in safe_keywords):
                await browser.close()
                return "Safe"
            tbody_xpath = '//*[@id="app"]/div/div[2]/section/div[1]/article[2]/table/tbody'
            if await page.locator(f'xpath={tbody_xpath}//tr').count() > 0:
                rows = await page.locator(f'xpath={tbody_xpath}//tr').all()
                for row in rows:
                    cols = await row.locator('td').all()
                    if len(cols) >= 2:
                        status_text = (await cols[1].inner_text()).lower()
                        if any(key in status_text for key in danger_keywords):
                            await browser.close()
                            return "Dangerous"
                        if any(key in status_text for key in safe_keywords):
                            await browser.close()
                            return "Safe"
            await browser.close()
            return "Not found"
        except Exception as e:
            return "Not found"

async def comprehensive_domain_check(raw_domain):
    clean_dom = clean_domain(raw_domain)
    full_url = format_url(raw_domain)
    task_whois = asyncio.to_thread(get_domain_age_sync, clean_dom)
    task_cld = check_chongluadao(clean_dom)
    task_dom = analyze_dom_and_phash(full_url)
    age_days, cld_status, dom_analysis = await asyncio.gather(task_whois, task_cld, task_dom)
    overall_status = "Safe"
    reasons = []
    if cld_status == "Dangerous":
        overall_status = "Dangerous"
        reasons.append("Flagged as phishing/malicious by ChongLuaDao.vn.")
    if len(dom_analysis["risk_flags"]) > 0:
        overall_status = "Dangerous"
        reasons.append("High-risk website source code detected (Stealing forms, hidden Iframes, phishing text...).")
    if overall_status != "Dangerous":
        if 0 <= age_days < 30:
            overall_status = "Warning"
            reasons.append(f"Domain is too new (created only {age_days} days ago), high risk.")
        elif age_days == -1:
            overall_status = "Warning"
            reasons.append("Domain intentionally hides WHOIS registration information or it cannot be retrieved.")
        if cld_status == "Not found" and overall_status == "Warning":
            reasons.append("Website lacks community reputation data.")
    if overall_status == "Safe":
        if cld_status == "Safe": reasons.append("Domain is certified safe.")
        else: reasons.append("No phishing signs detected in source code and domain data.")
    return {
        "domain": clean_dom,
        "overall_status": overall_status,
        "overall_reasons": reasons,
        "age_days": age_days,
        "3rd_party_status": cld_status,
        "suspicious_forms": dom_analysis["suspicious_forms_count"],
        "risk_flags": dom_analysis["risk_flags"]
    }

# ... (Phần WebSocket Server và Tunnel Manager GIỮ NGUYÊN) ...

@app.websocket("/ws")
async def websocket_endpoint(websocket: WebSocket):
    await websocket.accept()
    print("\n📱 Client đã kết nối thành công!")
    frame_count = 0
    batch_scores = []
    try:
        while True:
            message = await websocket.receive()
            if "bytes" in message:
                data = message["bytes"]
                nparr = np.frombuffer(data, np.uint8)
                frame = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
                if frame is not None:
                    frame_count += 1
                    _, confidence = await asyncio.to_thread(predict_deepfake, frame)
                    label = "DEEPFAKE" if confidence > 80.0 else "REAL"
                    print(f"\n[ANH #{frame_count}] {label} - {confidence:.2f}%")
                    frame_rgb_display = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                    img_pil = PILImage.fromarray(frame_rgb_display)
                    img_pil.thumbnail((320, 320))
                    display(img_pil)
                    if confidence > 0: batch_scores.append(confidence)
                    if len(batch_scores) >= 5:
                        avg_score = sum(batch_scores) / len(batch_scores)
                        agg_label = "DEEPFAKE" if avg_score > 80.0 else "REAL"
                        await websocket.send_json({
                            "type": "DEEPFAKE_RESULT", "result": agg_label, "score": round(avg_score, 2),
                            "average_score": round(avg_score, 2), "frame_count": frame_count, "batch_size": len(batch_scores)
                        })
                        batch_scores.clear()
                else:
                    await websocket.send_json({"type": "ERROR", "message": "Decode image failed"})
            elif "text" in message:
                domain = message["text"].strip()
                full_report = await comprehensive_domain_check(domain)
                await websocket.send_json({"type": "FRAUD_CHECK_RESULT", "data": full_report})
    except Exception: pass

def tunnel_manager():
    log_file = os.path.join(BASE_DIR, "cloudflared.log")
    while True:
        if os.path.exists(log_file): os.remove(log_file)
        command = f"cloudflared tunnel --url http://127.0.0.1:8000 > {log_file} 2>&1"
        proc = subprocess.Popen(command, shell=True)
        start_wait = time.time()
        while time.time() - start_wait < 30:
            if os.path.exists(log_file):
                with open(log_file, "r") as f:
                    content = f.read()
                    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
                    if match:
                        ws_url = f"{match.group().replace('https://', 'wss://')}/ws"
                        print(f"🔗 URL CÔNG KHAI: {match.group()}")
                        try: requests.put(FIREBASE_DB_URL, json={"ws_url": ws_url, "time": time.ctime()})
                        except: pass
                        break
            time.sleep(1)
        monitor_start = time.time()
        while time.time() - monitor_start < RESTART_INTERVAL:
            if proc.poll() is not None: break
            time.sleep(5)
        try: os.kill(proc.pid, signal.SIGTERM)
        except: pass
        time.sleep(2)

# ==========================================
# 6. CHAY HE THONG
# ==========================================
if model is None or face_detector is None:
    load_models()
else:
    print("✅ Models đã sẵn sàng (skip reload)")

_tunnel_started = globals().get("_tunnel_started", False)
if not _tunnel_started:
    threading.Thread(target=tunnel_manager, daemon=True).start()
    globals()["_tunnel_started"] = True
    print("🚀 Tunnel manager đã được khởi động")

config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="error")
server = uvicorn.Server(config)
server.install_signal_handlers = lambda: None
await server.serve()